# Lecture 19 – Cloud Computing and Storage (Part II)
**ISA 414 · Trey Patrick**

SaaS hands-on: calling cloud-hosted models instead of running them locally.
1. Text analytics – sentiment with `finiteautomata/bertweet-base-sentiment-analysis` (Hugging Face)
2. Image analytics – object detection with `facebook/detr-resnet-101` (Hugging Face)
3. Psychological assessment – Humantic AI (REST GET → JSON)

**Setup:** put the `.env` file from Canvas in this same folder (it's git-ignored, so it won't get pushed). Optional images from Canvas: `Brad-Pitt.jpg`, `iamaperson.jpg`.

## Lecture notes – service models
| Model | What you rent | Example from class |
|---|---|---|
| IaaS | A machine | — |
| PaaS | An environment | MongoDB Atlas |
| SaaS | A ready-made app | ChatGPT, Hugging Face models, Humantic AI |

**On-premise vs cloud – 4 considerations**
- **Cost:** on-prem = upfront hardware (sized by *peak analysis*), real estate, electricity, staff. Cloud = pay per use, cost calculators remove most uncertainty.
- **Demand:** if infrastructure would be underutilized (e.g. ISA 414 only runs 8 months/yr), cloud usually wins.
- **Skills:** do we have IT/security/DB/network people in-house? Cloud outsources maintenance – but do you trust the provider?
- **Security:** how does the provider protect data, and *where* is the cloud (geopolitics/legal)?

No one-size answer – hybrid is possible.

In [ ]:
# !pip install python-dotenv requests huggingface_hub pandas pillow
import os, requests, json
import pandas as pd
from dotenv import load_dotenv

load_dotenv()  # reads the .env file in this folder
HF_TOKEN = os.getenv("HF_TOKEN")
HUMANTIC_KEY = os.getenv("HUMANTIC_API_KEY")
print("HF token loaded:", HF_TOKEN is not None)
print("Humantic key loaded:", HUMANTIC_KEY is not None)

## SaaS Case Study #1 – Sentiment analysis (text)
Question: how did consumer sentiment toward Starbucks compare **before vs after** its pledge to hire refugees?

Flow: Facebook posts → submit to cloud model → model returns label + score → extract sentiment.

In [ ]:
from huggingface_hub import InferenceClient

client = InferenceClient(provider="hf-inference", api_key=HF_TOKEN)
SENT_MODEL = "finiteautomata/bertweet-base-sentiment-analysis"

result = client.text_classification("I love my morning Starbucks latte!", model=SENT_MODEL)
result  # labels: POS / NEU / NEG with scores

In [ ]:
# Swap in the Starbucks posts from the lecture notebook / Canvas data
posts = pd.DataFrame({
    "period": ["before", "before", "after", "after"],
    "text": [
        "Starbucks is my go-to coffee spot",
        "Pumpkin spice season at Starbucks!",
        "Never buying Starbucks again after this",
        "Proud of Starbucks for doing the right thing",
    ],
})

def top_label(text):
    preds = client.text_classification(text, model=SENT_MODEL)
    best = max(preds, key=lambda p: p.score)
    return best.label

posts["sentiment"] = posts["text"].apply(top_label)
posts

In [ ]:
# Compare sentiment distribution before vs after the pledge
pd.crosstab(posts["period"], posts["sentiment"], normalize="index").round(2)

**Reflection:** pre-trained models skip all the preprocessing (tokenizing, stopwords, etc.) – the whole thing is ~10 lines.
Do I trust the results? _…_

## SaaS Case Study #2 – Object detection (image)
Story: dating site **miamiohmerger.com** – every profile picture must contain a person.

Flow: user posts image → submitted to cloud model → returns detected objects (label, score, box).

In [ ]:
DETR_MODEL = "facebook/detr-resnet-101"

detections = client.object_detection("iamaperson.jpg", model=DETR_MODEL)
for d in detections:
    print(f"{d.label:<12} score={d.score:.3f}")

In [ ]:
def count_people(image_path, threshold=0.9):
    dets = client.object_detection(image_path, model=DETR_MODEL)
    return sum(1 for d in dets if d.label == "person" and d.score >= threshold)

for img in ["iamaperson.jpg", "Brad-Pitt.jpg"]:
    if os.path.exists(img):
        print(img, "→ people detected:", count_people(img))

**Reflection:** could build a whole product on this (also: detecting gory/explicit content, brands, celebrities, landmarks, handwriting).
Would I trust it? _…_

## SaaS Case Study #3 – Humantic AI (personality from LinkedIn)
Problem: sell to / hire someone more effectively based on their psychological profile.

Request: REST **GET** → Response: **JSON**. Get a key at https://api.humantic.ai/ (or use the one in `.env`).

In [ ]:
# Step 1: CREATE the analysis (first call can take a bit to process)
LINKEDIN_URL = "https://www.linkedin.com/in/ruiyun-xu-is"
headers = {"Content-Type": "application/json"}

create = requests.get(
    "https://api.humantic.ai/v1/user-profile/create",
    params={"apikey": HUMANTIC_KEY, "id": LINKEDIN_URL},
    headers=headers,
)
print(create.status_code, create.text[:300])


In [ ]:
# Step 2: FETCH the profile (same id used to create it)
fetch = requests.get(
    "https://api.humantic.ai/v1/user-profile",
    params={"apikey": HUMANTIC_KEY, "id": LINKEDIN_URL},
    headers=headers,
)
profile = fetch.json()
print(json.dumps(profile, indent=2)[:2000])


**Reflection:** no psychology background needed to build profiles – but it's a black box. Would I trust it for hiring decisions? _…_

## Summary
- Cloud tech improves the big-data process from infrastructure up to off-the-shelf software.
- Many SaaS services are black boxes → how much do you trust the provider, and could you do better with your own resources?
- **Homework 9:** implement the miamiohmerger.com check – is there a *single* person in the picture? (see Canvas)
- Next lecture: Hadoop